# CLIP-Guided Test-Time Optimization with Continuous Tokenizer (Hard VQ)

This notebook demonstrates CLIP-guided image editing using the Continuous Tokenizer with **hard vector quantization** (VQ-VAE style) instead of TiTok or SoftVQ.

## Key Difference from SoftVQ:
- **Hard VQ**: Uses discrete codebook lookup (argmin distance, like VQ-VAE)
- **SoftVQ**: Uses soft assignment with temperature (weighted sum)

Hard VQ has a **discrete bottleneck** similar to TiTok, which provides implicit regularization against adversarial CLIP optimization.

In [ ]:
try:
    import google.colab
    IN_COLAB = True
except:
    IN_COLAB = False

In [ ]:
import sys
if IN_COLAB:
    !git clone -q https://github.com/lukaslaobeyer/token-opt.git
    !pip install -q --progress-bar off jaxtyping open_clip_torch omegaconf
    sys.path.insert(0, "token-opt")
else:
    sys.path.insert(0, "..")

In [ ]:
import os
# Set this environment for deterministic execution
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

In [ ]:
import torch
# Enable for deterministic algorithms
torch.use_deterministic_algorithms(True, warn_only=False)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

from pathlib import Path
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.v2 as v2
import torchvision.transforms.v2.functional as tvf
from torchvision.datasets import ImageNet
from einops import rearrange

In [ ]:
from tto.test_time_opt import (
    TestTimeOpt,
    TestTimeOptConfig,
    CLIPObjective,
)

In [ ]:
#config variables
gpus = "8,9"

In [ ]:
# Set visible gpus
use_gpu = False
if use_gpu:

    os.environ["CUDA_VISIBLE_DEVICES"] = gpus
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device with ids: {os.environ['CUDA_VISIBLE_DEVICES']}")
    
else:
    device = torch.device("cpu")
    print("Using CPU device")

## Utils

In [ ]:
def load_img(path, device=None):
    if IN_COLAB:
        path = "./token-opt/notebooks" / Path(path)
    img = (1. / 255.) * torch.from_numpy(
        np.array(Image.open(path)).astype(np.float32)
    ).permute(2, 0, 1)
    img = tvf.resize(img, 256)
    img = tvf.center_crop(img, 256)
    img = img.unsqueeze(0)
    if device is not None:
        img = img.to(device)
    return img

def display_image(*tensors):
    tensors = [255. * t.squeeze() for t in tensors]
    img = Image.fromarray(rearrange(
        tensors, "b c h w -> h (b w) c"
    ).to("cpu", dtype=torch.uint8).numpy())
    display(img)

def opt_callback(info):
    if info.i % 50 == 0:
        print(f"i = {info.i}")
        print("  CLIP score =", "\t".join(
            map(lambda l: f"{-l:.3f}", info.loss))
        )
        imgs = tto.decode(info.tokens).clamp(0., 1.)
        display_image(*imgs)

# Set up the objective function

In [ ]:
# Use CLIP similarity maximization objective
objective = CLIPObjective(num_augmentations=8, cfg_scale=1.2)

# Set prompt
objective.prompt = [
    "a photo of a tiger",
    "a photo of a husky",
    "a photo of a sparrow",
]

# Optionally set a negative prompt
# Note: also need to set cfg_scale > 1 in CLIPObjective if using this!
objective.neg_prompt = "bad, low-res, unnatural"

# Configure test time optimization

**Key difference from SoftVQ**: We use `continuous_tokenizer:VQ` which uses **hard vector quantization**.

## Hard VQ vs SoftVQ:

### Hard VQ (this notebook):
- Uses **discrete** codebook lookup (argmin distance)
- Gradients flow through straight-through estimator
- Acts as **implicit regularizer** (like TiTok)
- Less prone to adversarial CLIP optimization

### SoftVQ:
- Uses **soft** assignment with temperature τ=0.07
- Smooth, fully differentiable gradients
- More flexible but requires stronger explicit regularization

## Optimization Settings:

Since Hard VQ has a discrete bottleneck (like TiTok), we can use **similar hyperparameters to the original notebook**:
- Standard learning rate (1e-1)
- Standard regularization (0.025)
- Can optimize either pre or post-quantization

**Recommended**: Optimize **post-quantization** to fully leverage the discrete constraint.

In [ ]:
tto_config = TestTimeOptConfig(
    # Use continuous tokenizer with Hard VQ model
    # Note: MAETok models use VQ, but currently no pretrained VQ-only checkpoints are available
    # You would need to train your own or use: "continuous_tokenizer:VQ:path/to/checkpoint"
    titok_checkpoint="continuous_tokenizer:VQ",
    
    # Optimize post-quantization (in discrete space) - recommended for Hard VQ
    # This leverages the discrete bottleneck for implicit regularization
    optimize_post_quantization_tokens=True,
    
    # VAE sampling (deterministic for reproducibility)
    vae_deterministic_sampling=True,
    
    # Optimization parameters (similar to original TiTok notebook)
    # Hard VQ's discrete bottleneck provides implicit regularization,
    # so we can use similar hyperparameters to TiTok
    num_iter=301,
    ema_decay=0.98,
    lr=1e-1,  # Standard learning rate (same as original)
    enable_amp=True,
    reg_weight=0.025,  # Standard regularization (same as original)
    #token_noise=1e-3,
    reg_type="seed",
)
tto = TestTimeOpt(tto_config, objective).to(device)

# Load seed images

In [ ]:
# Load seed image
img = torch.cat([
    load_img("ILSVRC2012_val_00008636.png", device),
    load_img("ILSVRC2012_val_00008636.png", device),
    load_img("ILSVRC2012_val_00010240.png", device),
], dim=0)

# Alternatively, initialize directly from given tokens (e.g. randomly
# sampled), but this is disabled when setting `seed_tokens = None`.
seed_tokens = None

# Run Optimization

In [ ]:
print("Seed")
display_image(*img)

# Run optimization
torch.manual_seed(0)
img_opt = tto(
    seed=img if seed_tokens is None else None,
    seed_tokens=seed_tokens,
    callback=opt_callback
)

## Model Configuration Details

The Continuous Tokenizer with Hard VQ uses:
- **Model Type**: VQ (hard vector quantization, VQ-VAE style)
- **Quantization**: Discrete codebook lookup (argmin distance)
- **Gradient Flow**: Straight-through estimator (like TiTok)
- **Encoder**: ViT (DINOv2 pretrained, configurable)
- **Decoder**: ViT
- **Image Size**: 256x256

### Key Differences from Other Tokenizers:

#### Hard VQ (this notebook):
1. **Discrete bottleneck**: Tokens snapped to nearest codebook entries
2. **Implicit regularization**: Quantization prevents adversarial patterns
3. **Straight-through gradients**: Approximate but stable
4. **Similar to TiTok**: Can use similar optimization hyperparameters

#### SoftVQ:
1. **Soft assignment**: Weighted combination of codebook entries (τ=0.07)
2. **Smooth gradients**: Fully differentiable but can be exploited
3. **Requires strong regularization**: Needs 4-6× stronger reg_weight
4. **More flexible**: Can represent points between codebook entries

#### TiTok:
1. **Cross-attention encoder**: Learnable query tokens
2. **Smaller latent space**: ~384 parameters vs 2048 for continuous tokenizers
3. **Pretrained on ImageNet**: Codebook optimized for natural images
4. **Different architecture**: Not ViT-based

### Expected Behavior:

Hard VQ should behave **similarly to TiTok** because:
- Both have discrete bottlenecks
- Both use straight-through gradient estimators
- Both provide implicit regularization against adversarial optimization

**CLIP scores** should be in the **0.13-0.25 range** (similar to TiTok), not the inflated 0.35+ range seen with under-regularized SoftVQ.

### Note on Checkpoints:

Currently, there are no standalone pretrained VQ checkpoints available on HuggingFace. The MAETok models use VQ internally but are designed for different purposes. To use this notebook effectively, you would need to either:
1. Train your own VQ tokenizer using the provided training scripts
2. Use MAETok checkpoints (would require adapting the wrapper)
3. Use SoftVQ or TiTok instead for pretrained models